### ARO-ALO Histogramlı Colon_ACA

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import io, filters, metrics
from mealpy.swarm_based import ARO, ALO

from mealpy.utils.space import FloatVar
import pandas as pd
from scipy.stats import friedmanchisquare, rankdata
from scipy.stats import chi2
import os
import glob
import warnings
warnings.filterwarnings('ignore')

class HybridARO_ALO:
    """
    Adaptif ARO-ALO Hibrit Optimizasyon Algoritması
    """
    def __init__(self, epoch=50, pop_size=30, switch_ratio=0.6):
        """
        Parameters:
        epoch: int - Toplam iterasyon sayısı
        pop_size: int - Popülasyon büyüklüğü
        switch_ratio: float - ARO'nun çalışacağı oran (0-1 arası)
        """
        self.epoch = epoch
        self.pop_size = pop_size
        self.switch_ratio = switch_ratio
        self.ARO_epochs = int(epoch * switch_ratio)
        self.ALO_epochs = epoch - self.ARO_epochs

    def solve(self, problem_dict):
        """
        Adaptif hibrit optimizasyon çözümü
        """
        print(f"Adaptif Hibrit: ARO ({self.ARO_epochs} epoch) + ALO ({self.ALO_epochs} epoch)")

        # 1. Aşama: ARO ile keşif
        if self.ARO_epochs > 0:
            ARO_model = ARO.OriginalARO(epoch=self.ARO_epochs, pop_size=self.pop_size)
            ARO_result = ARO_model.solve(problem_dict)
            best_position = ARO_result.solution
            best_fitness = ARO_result.target.fitness
            print(f"ARO Aşaması Tamamlandı - En İyi Fitness: {best_fitness:.6f}")
        else:
            # Eğer ARO epoch'u 0 ise rastgele başlangıç
            bounds = problem_dict["bounds"]
            if isinstance(bounds, list):
                best_position = np.array([np.random.uniform(b.lb, b.ub) for b in bounds])
            else:
                best_position = np.random.uniform(bounds.lb, bounds.ub)
            best_fitness = problem_dict["obj_func"](best_position)

        # 2. Aşama: ALO ile sömürü (ARO'nun en iyi çözümünden başlayarak)
        if self.ALO_epochs > 0:
            # ALO için özel problem tanımı - en iyi çözümle başlatmak için
            class CustomALO(ALO.OriginalALO):
                def __init__(self, best_solution, **kwargs):
                    super().__init__(**kwargs)
                    self.best_initial_solution = best_solution

                def initialization(self):
                    super().initialization()
                    # İlk bireyi ARO'nun en iyi çözümü ile değiştir
                    if len(self.pop) > 0:
                        self.pop[0].solution = self.best_initial_solution.copy()
                        self.pop[0].target = self.problem.get_target(self.pop[0].solution)

            ALO_model = CustomALO(
                best_solution=best_position,
                epoch=self.ALO_epochs,
                pop_size=self.pop_size
            )
            ALO_result = ALO_model.solve(problem_dict)

            # En iyi sonucu güncelle
            if ALO_result.target.fitness < best_fitness:
                best_position = ALO_result.solution
                best_fitness = ALO_result.target.fitness
                print(f"ALO İyileştirme Sağladı - Yeni En İyi Fitness: {best_fitness:.6f}")
            else:
                print(f"ARO Sonucu Daha İyi Kaldı - En İyi Fitness: {best_fitness:.6f}")

        # Sonucu mealpy formatında döndür
        class HybridResult:
            def __init__(self, solution, fitness):
                self.solution = solution
                self.target = type('Target', (), {'fitness': fitness})()

        return HybridResult(best_position, best_fitness)

class MultiLevelThresholding:
    def __init__(self, image, num_thresholds=3):
        """
        Medical image multi-level thresholding using Hybrid ARO-ALO

        Parameters:
        image: numpy array - Input medical image
        num_thresholds: int - Number of threshold levels
        """
        self.image = image.astype(np.float64)
        self.num_thresholds = num_thresholds
        self.height, self.width = image.shape
        self.histogram = np.histogram(image, bins=256, range=(0, 255))[0]

    def otsu_fitness(self, thresholds):
        """
        Otsu's method fitness function for multi-level thresholding
        """
        thresholds = np.sort(thresholds)
        thresholds = np.clip(thresholds, 0, 255)

        # Add boundary thresholds
        full_thresholds = [0] + list(thresholds) + [255]

        total_pixels = self.height * self.width
        total_mean = np.sum(np.arange(256) * self.histogram) / total_pixels

        between_class_variance = 0

        for i in range(len(full_thresholds) - 1):
            start_idx = int(full_thresholds[i])
            end_idx = int(full_thresholds[i + 1])

            # Class probability
            w_i = np.sum(self.histogram[start_idx:end_idx]) / total_pixels

            if w_i > 0:
                # Class mean
                class_sum = np.sum(np.arange(start_idx, end_idx) * self.histogram[start_idx:end_idx])
                mu_i = class_sum / (w_i * total_pixels) if (w_i * total_pixels) > 0 else 0

                # Between-class variance contribution
                between_class_variance += w_i * (mu_i - total_mean) ** 2

        return -between_class_variance  # Negative because we want to maximize

    def apply_thresholding(self, thresholds):
        """
        Apply multi-level thresholding to the image
        """
        thresholds = np.sort(thresholds)
        segmented = np.zeros_like(self.image)

        # Apply thresholds
        segmented[self.image <= thresholds[0]] = 0
        for i in range(len(thresholds) - 1):
            mask = (self.image > thresholds[i]) & (self.image <= thresholds[i + 1])
            segmented[mask] = (i + 1) * (255 // (len(thresholds) + 1))
        segmented[self.image > thresholds[-1]] = 255

        return segmented.astype(np.uint8)

    def optimize_thresholds(self, pop_size=25, epochs=4, switch_ratio=0.6):
        """
        Optimize thresholds using Hybrid ARO-ALO

        Parameters:
        pop_size: int - Population size
        epochs: int - Total number of epochs
        switch_ratio: float - Ratio for ARO phase (0-1)
        """
        # Define bounds using FloatVar for mealpy v3.0+
        bounds = [FloatVar(lb=1.0, ub=255.0, name=f"threshold_{i}")
                  for i in range(self.num_thresholds)]

        # Define problem dictionary for mealpy
        problem_dict = {
            "bounds": bounds,
            "minmax": "min",  # We're minimizing (fitness function returns negative value)
            "obj_func": self.otsu_fitness
        }

        # Initialize Hybrid ARO-ALO
        hybrid_model = HybridARO_ALO(
            epoch=epochs,
            pop_size=pop_size,
            switch_ratio=switch_ratio
        )

        # Solve optimization problem
        g_best = hybrid_model.solve(problem_dict)

        return np.sort(g_best.solution), -g_best.target.fitness

def calculate_psnr(original, segmented):
    """Calculate Peak Signal-to-Noise Ratio"""
    mse = np.mean((original - segmented) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

def calculate_gradient_magnitude(image):
    gx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    gm = np.sqrt(gx ** 2 + gy ** 2)
    return gm

def calculate_phase_congruency_approx(image):
    image_blur = cv2.GaussianBlur(image, (5, 5), 1.0)
    gx = cv2.Sobel(image_blur, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(image_blur, cv2.CV_64F, 0, 1, ksize=3)
    gm = np.sqrt(gx ** 2 + gy ** 2)
    pc = gm / (gm + 10)  # epsilon to avoid division by zero
    return pc

def calculate_fsim(original, test):
    original = original.astype(np.float64)
    test = test.astype(np.float64)

    if original.max() > 1.0:
        original /= 255.0
        test /= 255.0

    # 1. Phase Congruency
    PC1 = calculate_phase_congruency_approx(original)
    PC2 = calculate_phase_congruency_approx(test)

    # 2. Gradient Magnitude
    GM1 = calculate_gradient_magnitude(original)
    GM2 = calculate_gradient_magnitude(test)

    # 3. Similarity Maps
    T1 = 0.01
    T2 = 0.02

    SPC = (2 * PC1 * PC2 + T1) / (PC1 ** 2 + PC2 ** 2 + T1)
    SGM = (2 * GM1 * GM2 + T2) / (GM1 ** 2 + GM2 ** 2 + T2)

    # 4. Combined Similarity
    PCm = np.maximum(PC1, PC2)
    FSIM_map = SPC * SGM
    numerator = np.sum(FSIM_map * PCm)
    denominator = np.sum(PCm)

    fsim_value = numerator / (denominator + 1e-10)
    return fsim_value


def calculate_ssim(original, segmented):
    """Calculate Structural Similarity Index"""
    return metrics.structural_similarity(original, segmented, data_range=255)

def friedman_ranking_test(data_matrix):
    """
    Perform Friedman ranking test
    data_matrix: rows are observations, columns are treatments/methods
    """
    n_observations, n_treatments = data_matrix.shape

    # Rank each row
    ranks = np.zeros_like(data_matrix)
    for i in range(n_observations):
        ranks[i] = rankdata(-data_matrix[i])  # Negative for descending order (higher is better)

    # Calculate rank sums
    rank_sums = np.sum(ranks, axis=0)

    # Friedman test statistic
    chi2_stat = (12 / (n_observations * n_treatments * (n_treatments + 1))) * \
                np.sum(rank_sums**2) - 3 * n_observations * (n_treatments + 1)

    # Degrees of freedom
    df = n_treatments - 1

    # P-value
    p_value = 1 - chi2.cdf(chi2_stat, df)

    return chi2_stat, p_value, rank_sums

def load_medical_images(healthy_folder,
                        max_images_per_class=6):
    """
    Load medical images from healthy and diseased folders
    """
    images = []
    labels = []
    filenames = []

    # Supported image extensions
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']

    # Load healthy images
    print(f"Loading healthy images from: {healthy_folder}")
    healthy_count = 0
    if os.path.exists(healthy_folder):
        for ext in extensions:
            for img_path in glob.glob(os.path.join(healthy_folder, ext)):
                if healthy_count >= max_images_per_class:
                    break
                try:
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        # Resize image to standard size for consistency
                        img = cv2.resize(img, (256,256))
                        images.append(img)
                        labels.append('Healthy')
                        filenames.append(os.path.basename(img_path))
                        healthy_count += 1
                        print(f"  Loaded: {os.path.basename(img_path)}")
                except Exception as e:
                    print(f"  Error loading {img_path}: {e}")
            if healthy_count >= max_images_per_class:
                break
    else:
        print(f"  Warning: Healthy folder not found: {healthy_folder}")

    if len(images) == 0:
        print("No images found! Please check your folder paths.")
        return None, None, None

    return images, labels, filenames

def main_analysis(healthy_folder="./healthy",
                  max_images_per_class=6):
    """
    Main analysis function with Hybrid ARO-ALO optimization
    """
    print("Medical Image Multi-level Thresholding Analysis with Hybrid ARO-ALO")
    print("=" * 70)

    # Load images from folders
    images, labels, filenames = load_medical_images(
        healthy_folder,
        max_images_per_class
    )

    if images is None:
        print("No images loaded. Exiting analysis.")
        return None, None, None

    # Different threshold levels to test
    threshold_levels = [2, 3, 4, 5, 6, 7,8,9,10,11,12]

    # Hybrid algorithm parameters
    hybrid_params = {
        'pop_size': 30,
        'epochs': 50,
        'switch_ratio': 0.6  # %60 ARO, %40 ALO
    }

    # Results storage
    results = {
        'Image': [],
        'Label': [],
        'Filename': [],
        'Thresholds': [],
        'PSNR': [],
        'SSIM': [],
        'FSIM': [],
        'Optimal_Thresholds': [],
        'Fitness_Value': [],
        'Algorithm': []
    }

    # Calculate number of subplot rows and columns
    n_images = len(images)
    n_cols = min(2, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols

    plt.figure(figsize=(n_cols * 6, n_rows * 3))

    for img_idx, (image, label, filename) in enumerate(zip(images, labels, filenames)):
        print(f"\nProcessing {filename} ({label})...")

        for thresh_level in threshold_levels:
            print(f"  Threshold levels: {thresh_level}")

            # Initialize thresholding with Hybrid ARO-ALO
            mt = MultiLevelThresholding(image, num_thresholds=thresh_level)

            # Optimize thresholds using Hybrid ARO-ALO
            optimal_thresholds, fitness_value = mt.optimize_thresholds(
                pop_size=hybrid_params['pop_size'],
                epochs=hybrid_params['epochs'],
                switch_ratio=hybrid_params['switch_ratio']
            )

            # Apply thresholding
            segmented = mt.apply_thresholding(optimal_thresholds)

            # Calculate metrics
            psnr = calculate_psnr(image, segmented)
            ssim = calculate_ssim(image, segmented)
            fsim = calculate_fsim(image, segmented)

            # Store results
            results['Image'].append(f"{label}_{img_idx+1}")
            results['Label'].append(label)
            results['Filename'].append(filename)
            results['Thresholds'].append(thresh_level)
            results['PSNR'].append(psnr)
            results['SSIM'].append(ssim)
            results['FSIM'].append(fsim)
            results['Optimal_Thresholds'].append(optimal_thresholds)
            results['Fitness_Value'].append(fitness_value)
            results['Algorithm'].append('Hybrid_ARO-ALO')

            print(f"    PSNR: {psnr:.2f}, SSIM: {ssim:.4f}, FSIM: {fsim:.4f}")
            print(f"    Optimal thresholds: {[f'{t:.1f}' for t in optimal_thresholds]}")

        # Visualize results for 3-level thresholding
        # Tek sefer optimizasyon - sonuçları sakla
        if img_idx == 0:  # İlk defa çalıştırılıyorsa storage listesi oluştur
            optimization_results = []

        # Visualize results for 3-level thresholding
        mt_viz = MultiLevelThresholding(image, num_thresholds=12)
        opt_thresh, _ = mt_viz.optimize_thresholds(
            pop_size=30,
            epochs=100,
            switch_ratio=0.3
        )
        segmented_viz = mt_viz.apply_thresholding(opt_thresh)

        # Sonuçları sakla
        optimization_results.append({
            'image': image,
            'segmented': segmented_viz,
            'thresholds': opt_thresh,
            'filename': filename,
            'label': label
        })

        # Plot original and segmented
        plt.subplot(n_rows, n_cols * 2, img_idx * 2 + 1)
        plt.imshow(image, cmap='plasma')
        plt.title('Original')
        plt.axis('off')

        plt.subplot(n_rows, n_cols * 2, img_idx * 2 + 2)
        im = plt.imshow(segmented_viz, cmap='plasma')
        plt.title('Hybrid ARO-ALO')
        plt.axis('off')
        # Scale bar ekleme
        plt.colorbar(im, ax=plt.gca(), fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


    # Calculate number of subplot rows and columns
    n_images = len(images)
    n_cols = min(1, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols

    # Ayrı histogram analizi - saklanan sonuçları kullan
    print("\n" + "="*60)
    print("HISTOGRAM ANALYSIS")
    print("="*60)

    plt.figure(figsize=(n_cols * 6, n_rows * 3))
    image_number_count=1
    for idx, result in enumerate(optimization_results):
        image = result['image']
        segmented_viz = result['segmented']
        opt_thresh = result['thresholds']
        filename = result['filename']
        label = result['label']


        # Original 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 1)

        # Gradyan hesaplama (X ve Y yönünde)
        grad_x = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
        gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)

        # 2D histogram: Pixel değeri vs Gradyan büyüklüğü
        h, xedges, yedges = np.histogram2d(
            image.flatten(),
            gradient_magnitude.flatten(),
            bins=[64, 64],
            range=[[0, 255], [0, gradient_magnitude.max()]]
        )

        # 2D histogram görselleştirme
        extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
        im1 = plt.imshow(h.T, extent=extent, origin='lower', cmap='Blues', aspect='auto')
        plt.colorbar(im1, fraction=0.046, pad=0.04)
        plt.xlabel('Pixel Value')
        plt.ylabel('Gradient Magnitude')
        plt.title(f'Image {image_number_count}-2D Histogram\n) - Pixel vs Gradient')

        # Threshold çizgilerini ekleme
        for i, thresh in enumerate(opt_thresh):
            plt.axvline(x=thresh, color='red', linestyle='--', alpha=0.8, linewidth=2)

        # Segmented 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 2)

        # Segmented image için gradyan hesaplama
        grad_x_seg = cv2.Sobel(segmented_viz.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        grad_y_seg = cv2.Sobel(segmented_viz.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
        gradient_magnitude_seg = np.sqrt(grad_x_seg**2 + grad_y_seg**2)

        # 2D histogram: Pixel değeri vs Gradyan büyüklüğü (segmented)
        h_seg, xedges_seg, yedges_seg = np.histogram2d(
            segmented_viz.flatten(),
            gradient_magnitude_seg.flatten(),
            bins=[64, 64],
            range=[[0, 255], [0, gradient_magnitude_seg.max() if gradient_magnitude_seg.max() > 0 else 1]]
        )

        # 2D histogram görselleştirme (segmented)
        extent_seg = [xedges_seg[0], xedges_seg[-1], yedges_seg[0], yedges_seg[-1]]
        im2 = plt.imshow(h_seg.T, extent=extent_seg, origin='lower', cmap='Reds', aspect='auto')
        plt.colorbar(im2, fraction=0.046, pad=0.04)
        plt.xlabel('Pixel Value')
        plt.ylabel('Gradient Magnitude')
        plt.title(f'Segmented 2D Histogram\n-Hybrid ARO-ALO')
        image_number_count+=1

        # Threshold çizgilerini ekleme
        for i, thresh in enumerate(opt_thresh):
            plt.axvline(x=thresh, color='green', linestyle='--', alpha=0.8, linewidth=2,
                      label=f'T{i+1}: {thresh:.1f}' if idx == 0 else "")

        if idx == 0 and len(opt_thresh) > 0:
            plt.legend(fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()

    # Ek olarak: Pixel yoğunluk korelasyon analizi
    print("\n" + "="*60)
    print("PIXEL INTENSITY CORRELATION ANALYSIS")
    print("="*60)

    plt.figure(figsize=(n_cols * 12, n_rows * 6))

    image_number_count=1
    for idx, result in enumerate(optimization_results):
        image = result['image']
        segmented_viz = result['segmented']
        filename = result['filename']
        label = result['label']


        # Original vs Segmented pixel correlation
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 1)

        # Scatter plot: Original vs Segmented pixel values
        sample_indices = np.random.choice(image.size, min(5000, image.size), replace=False)
        orig_sample = image.flatten()[sample_indices]
        seg_sample = segmented_viz.flatten()[sample_indices]

        plt.hexbin(orig_sample, seg_sample, gridsize=50, cmap='Blues', alpha=0.7)
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.xlabel('Original Pixel Value',fontdict={'size': 16})
        plt.ylabel('Segmented Pixel Value',fontdict={'size': 16})
        plt.title(f'Test Image {image_number_count} - Pixel Correlation\n - Original vs Segmented', fontdict={'size': 16})
        plt.plot([0, 255], [0, 255], 'r--', alpha=0.5, label='Perfect Correlation')
        plt.legend(fontsize="16")

        # Komşuluk analizi 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 2)

        # Komşu pixel ilişkileri (horizontal)
        left_pixels = image[:, :-1].flatten()
        right_pixels = image[:, 1:].flatten()

        # 2D histogram: Sol pixel vs Sağ pixel
        h_neighbor, x_edges, y_edges = np.histogram2d(
            left_pixels, right_pixels,
            bins=[64, 64],
            range=[[0, 255], [0, 255]]
        )

        extent_neighbor = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
        im_neighbor = plt.imshow(h_neighbor.T, extent=extent_neighbor, origin='lower',
                              cmap='Greens', aspect='auto')
        plt.colorbar(im_neighbor, fraction=0.046, pad=0.04)
        plt.xlabel('Left Neighbor Pixel Value',fontdict={'size': 16})
        plt.ylabel('Right Neighbor Pixel Value',fontdict={'size': 16})
        plt.title(f'Test Image {image_number_count} - Spatial Correlation\n - Neighbor Pixel Analysis', fontdict={'size': 16})
        image_number_count+=1

    plt.tight_layout()
    plt.show()


    from scipy.stats import pearsonr
    # Orijinal vs Segment edilmiş piksel korelasyonu
    corr_orig_seg, _ = pearsonr(orig_sample, seg_sample)
    print(f"Original vs Segmented Pixel Pearson Correlation: {corr_orig_seg:.4f}")
    # Komşu pikseller arasındaki korelasyon (yatay)
    corr_neighbor, _ = pearsonr(left_pixels, right_pixels)
    print(f"Neighbor Pixel Pearson Correlation: {corr_neighbor:.4f}")


    from scipy.stats import spearmanr
    spearman_corr_orig_seg, _ = spearmanr(orig_sample, seg_sample)
    spearman_corr_neighbor, _ = spearmanr(left_pixels, right_pixels)
    print(f"Original vs Segmented Pixel Spearman Correlation: {spearman_corr_orig_seg:.4f}")
    print(f"Neighbor Pixel Spearman Correlation: {spearman_corr_neighbor:.4f}")

    from sklearn.metrics import mutual_info_score
    def mutual_info(x, y, bins=64):
        c_xy = np.histogram2d(x, y, bins)[0]
        mi = mutual_info_score(None, None, contingency=c_xy)
        return mi
    mi_orig_seg = mutual_info(orig_sample, seg_sample)
    mi_neighbor = mutual_info(left_pixels, right_pixels)
    print(f"Original vs Segmented Pixel Mutual Information: {mi_orig_seg:.4f}")
    print(f"Neighbor Pixel Mutual Information: {mi_neighbor:.4f}")

    # Create results DataFrame
    df = pd.DataFrame(results)
    print("\n" + "="*100)
    print("DETAILED RESULTS - HYBRID ARO-ALO OPTIMIZATION")
    print("="*100)
    print(df.to_string(index=False))

    # Statistical Analysis
    print("\n" + "="*100)
    print("HYBRID ARO-ALO PERFORMANCE ANALYSIS")
    print("="*100)

    # Performance comparison between healthy and diseased images
    healthy_df = df[df['Label'] == 'Healthy']
    #diseased_df = df[df['Label'] == 'Diseased']

    if len(healthy_df) > 0: #and len(diseased_df) > 0:
        print("\n--- HEALTHY vs DISEASED COMPARISON (Hybrid ARO-ALO) ---")

        healthy_psnr_avg = healthy_df.groupby('Thresholds')['PSNR'].mean()
        #diseased_psnr_avg = diseased_df.groupby('Thresholds')['PSNR'].mean()
        healthy_ssim_avg = healthy_df.groupby('Thresholds')['SSIM'].mean()
        #diseased_ssim_avg = diseased_df.groupby('Thresholds')['SSIM'].mean()
        healthy_fsim_avg = healthy_df.groupby('Thresholds')['FSIM'].mean()

        comparison_df = pd.DataFrame({
            'Thresholds': threshold_levels,
            'Healthy_PSNR': healthy_psnr_avg.values,
            #'Diseased_PSNR': diseased_psnr_avg.values,
            'Healthy_SSIM': healthy_ssim_avg.values,
            #'Diseased_SSIM': diseased_ssim_avg.values,
            'Healthy_FSIM': healthy_fsim_avg.values,
            #'PSNR_Improvement': healthy_psnr_avg.values - diseased_psnr_avg.values,
            #'SSIM_Improvement': healthy_ssim_avg.values - diseased_ssim_avg.values
        })

        print("\nHybrid ARO-ALO Performance Comparison:")
        print(comparison_df.round(4))

    # Friedman test analysis
    unique_images = df['Image'].unique()
    if len(unique_images) >= 3:

        psnr_matrix = np.zeros((len(unique_images), len(threshold_levels)))
        ssim_matrix = np.zeros((len(unique_images), len(threshold_levels)))
        fsim_matrix = np.zeros((len(unique_images), len(threshold_levels)))

        for i, img_name in enumerate(unique_images):
            img_data = df[df['Image'] == img_name]
            if len(img_data) == len(threshold_levels):
                psnr_matrix[i] = img_data['PSNR'].values
                ssim_matrix[i] = img_data['SSIM'].values
                fsim_matrix[i] = img_data['FSIM'].values

        # Friedman test for PSNR
        chi2_psnr, p_psnr, ranks_psnr = friedman_ranking_test(psnr_matrix)
        print(f"\nHybrid ARO-ALO PSNR Analysis:")
        print(f"Chi-square statistic: {chi2_psnr:.4f}")
        print(f"P-value: {p_psnr:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_psnr}")

        if p_psnr < 0.05:
            print("Significant difference found between threshold levels (PSNR)")
            best_psnr_level = threshold_levels[np.argmin(ranks_psnr)]
            print(f"Best performing threshold level (PSNR): {best_psnr_level}")
        else:
            print("No significant difference between threshold levels (PSNR)")

        # Friedman test for SSIM
        chi2_ssim, p_ssim, ranks_ssim = friedman_ranking_test(ssim_matrix)
        print(f"\nHybrid ARO-ALO SSIM Analysis:")
        print(f"Chi-square statistic: {chi2_ssim:.4f}")
        print(f"P-value: {p_ssim:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_ssim}")

        if p_ssim < 0.05:
            print("Significant difference found between threshold levels (SSIM)")
            best_ssim_level = threshold_levels[np.argmin(ranks_ssim)]
            print(f"Best performing threshold level (SSIM): {best_ssim_level}")
        else:
            print("No significant difference between threshold levels (SSIM)")


        # Friedman test for FSIM
        chi2_fsim, p_fsim, ranks_fsim = friedman_ranking_test(fsim_matrix)
        print(f"\nHybrid ARO-ALO FSIM Analysis:")
        print(f"Chi-square statistic: {chi2_fsim:.4f}")
        print(f"P-value: {p_fsim:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_fsim}")
        if p_fsim < 0.05:
            print("Significant difference found between threshold levels (FSIM)")
            best_fsim_level = threshold_levels[np.argmin(ranks_fsim)]
            print(f"Best performing threshold level (FSIM): {best_fsim_level}")
        else:
            print("No significant difference between threshold levels (FSIM)")

    else:
        print(f"Not enough images ({len(unique_images)}) for Friedman test.")
        chi2_psnr, p_psnr, ranks_psnr = None, None, None
        chi2_ssim, p_ssim, ranks_ssim = None, None, None
        chi2_fsim, p_fsim, ranks_fsim = None, None, None

    # Summary Statistics
    print("\n" + "="*100)
    print("HYBRID ARO-ALO SUMMARY STATISTICS")
    print("="*100)

    if len(df) > 0:
        summary_stats = df.groupby(['Label', 'Thresholds']).agg({
            'PSNR': ['mean', 'std', 'min', 'max'],
            'SSIM': ['mean', 'std', 'min', 'max'],
            'FSIM': ['mean', 'std', 'min', 'max'],
            'Fitness_Value': ['mean', 'std', 'min', 'max']
        }).round(4)

        print(summary_stats)

        # Plot enhanced comparison charts
        plt.figure(figsize=(20, 12))

        # PSNR comparison
        plt.subplot(3, 4, 1)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            psnr_means = label_data.groupby('Thresholds')['PSNR'].mean()
            psnr_stds = label_data.groupby('Thresholds')['PSNR'].std()
            plt.errorbar(threshold_levels, psnr_means, yerr=psnr_stds,
                        marker='o', label=f'{label} (Hybrid ARO-ALO)',
                        linewidth=2, capsize=5)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average PSNR')
        plt.title('Hybrid ARO-ALO: PSNR Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # SSIM comparison
        plt.subplot(3, 4, 2)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            ssim_means = label_data.groupby('Thresholds')['SSIM'].mean()
            ssim_stds = label_data.groupby('Thresholds')['SSIM'].std()
            plt.errorbar(threshold_levels, ssim_means, yerr=ssim_stds,
                        marker='s', label=f'{label} (Hybrid ARO-ALO)',
                        linewidth=2, capsize=5)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average SSIM')
        plt.title('Hybrid ARO-ALO: SSIM Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)



        # FSIM comparison
        plt.subplot(3, 4, 3)
        for label in df['Label'].unique():
          label_data = df[df['Label'] == label]
          fsim_means = label_data.groupby('Thresholds')['FSIM'].mean()
          fsim_stds = label_data.groupby('Thresholds')['FSIM'].std()
          plt.errorbar(threshold_levels, fsim_means, yerr=fsim_stds,
                       marker='^', label=f'{label} (Hybrid ARO-ALO)',
                       linewidth=2, capsize=5)
          plt.xlabel('Threshold Levels')
          plt.ylabel('Average FSIM')
          plt.title('Hybrid ARO-ALO: FSIM Performance')
          plt.legend()
          plt.grid(True, alpha=0.3)

        # Fitness value comparison
        plt.subplot(3, 4, 4)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            fitness_means = label_data.groupby('Thresholds')['Fitness_Value'].mean()
            plt.plot(threshold_levels, fitness_means, marker='^',
                    label=f'{label} (Hybrid ARO-ALO)', linewidth=2)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average Fitness Value')
        plt.title('Hybrid ARO-ALO: Fitness Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Box plots
        plt.subplot(3, 4, 5)
        df.boxplot(column='PSNR', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('PSNR Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 6)
        df.boxplot(column='SSIM', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('SSIM Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 7)
        df.boxplot(column='FSIM', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('FSIM Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 8)
        df.boxplot(column='Fitness_Value', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('Fitness Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.tight_layout()
        plt.show()

    return df, (chi2_psnr, p_psnr, ranks_psnr), (chi2_ssim, p_ssim, ranks_ssim), (chi2_fsim, p_fsim, ranks_fsim)

if __name__ == "__main__":
    # Dataset Path
    healthy_folder = ""


    print("Starting Hybrid ARO-ALO Medical Image Segmentation Analysis...")
    print("Algorithm Configuration:")
    print("- Phase 1: ARO (60% of epochs) - Exploration")
    print("- Phase 2: ALO (40% of epochs) - Exploitation")
    print("- Population Size: 30")
    print("- Total Epochs: 100-150")
    print("-" * 60)

    # Analizi çalıştır
    results_df, psnr_test, ssim_test, fsim_test = main_analysis(
        healthy_folder=healthy_folder,

        max_images_per_class=100
    )

    if results_df is not None:
        print("\nHybrid ARO-ALO Analysis completed successfully!")
        print("Results saved in 'results_df' DataFrame")

        # Sonuçları CSV dosyasına kaydet
        results_df.to_csv('hybrid_ARO_ALO_segmentation_results.csv', index=False)
        print("Results also saved to 'hybrid_ARO_ALO_segmentation_results.csv'")

        # Algoritma performans özeti
        print("\n" + "="*80)
        print("HYBRID ARO-ALO ALGORITHM PERFORMANCE SUMMARY")
        print("="*80)

        avg_psnr = results_df['PSNR'].mean()
        avg_ssim = results_df['SSIM'].mean()
        avg_fsim = results_df['FSIM'].mean()
        avg_fitness = results_df['Fitness_Value'].mean()

        print(f"Overall Average PSNR: {avg_psnr:.4f}")
        print(f"Overall Average SSIM: {avg_ssim:.4f}")
        print(f"Overall Average FSIM: {avg_fsim:.4f}")
        print(f"Overall Average Fitness: {avg_fitness:.6f}")

        # En iyi performans gösteren threshold seviyesi
        best_threshold_psnr = results_df.groupby('Thresholds')['PSNR'].mean().idxmax()
        best_threshold_ssim = results_df.groupby('Thresholds')['SSIM'].mean().idxmax()
        best_threshold_fsim = results_df.groupby('Thresholds')['FSIM'].mean().idxmax()

        print(f"Best Threshold Level for PSNR: {best_threshold_psnr}")
        print(f"Best Threshold Level for SSIM: {best_threshold_ssim}")
        print(f"Best Threshold Level for FSIM: {best_threshold_fsim}")

    else:
        print("Analysis failed - no images were loaded.")

### ARO-ALO Histogramlı Colon_N

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import io, filters, metrics
from mealpy.swarm_based import ARO, ALO

from mealpy.utils.space import FloatVar
import pandas as pd
from scipy.stats import friedmanchisquare, rankdata
from scipy.stats import chi2
import os
import glob
import warnings
warnings.filterwarnings('ignore')

class HybridARO_ALO:
    """
    Adaptif ARO-ALO Hibrit Optimizasyon Algoritması
    """
    def __init__(self, epoch=50, pop_size=30, switch_ratio=0.6):
        """
        Parameters:
        epoch: int - Toplam iterasyon sayısı
        pop_size: int - Popülasyon büyüklüğü
        switch_ratio: float - ARO'nun çalışacağı oran (0-1 arası)
        """
        self.epoch = epoch
        self.pop_size = pop_size
        self.switch_ratio = switch_ratio
        self.ARO_epochs = int(epoch * switch_ratio)
        self.ALO_epochs = epoch - self.ARO_epochs

    def solve(self, problem_dict):
        """
        Adaptif hibrit optimizasyon çözümü
        """
        print(f"Adaptif Hibrit: ARO ({self.ARO_epochs} epoch) + ALO ({self.ALO_epochs} epoch)")

        # 1. Aşama: ARO ile keşif
        if self.ARO_epochs > 0:
            ARO_model = ARO.OriginalARO(epoch=self.ARO_epochs, pop_size=self.pop_size)
            ARO_result = ARO_model.solve(problem_dict)
            best_position = ARO_result.solution
            best_fitness = ARO_result.target.fitness
            print(f"ARO Aşaması Tamamlandı - En İyi Fitness: {best_fitness:.6f}")
        else:
            # Eğer ARO epoch'u 0 ise rastgele başlangıç
            bounds = problem_dict["bounds"]
            if isinstance(bounds, list):
                best_position = np.array([np.random.uniform(b.lb, b.ub) for b in bounds])
            else:
                best_position = np.random.uniform(bounds.lb, bounds.ub)
            best_fitness = problem_dict["obj_func"](best_position)

        # 2. Aşama: ALO ile sömürü (ARO'nun en iyi çözümünden başlayarak)
        if self.ALO_epochs > 0:
            # ALO için özel problem tanımı - en iyi çözümle başlatmak için
            class CustomALO(ALO.OriginalALO):
                def __init__(self, best_solution, **kwargs):
                    super().__init__(**kwargs)
                    self.best_initial_solution = best_solution

                def initialization(self):
                    super().initialization()
                    # İlk bireyi ARO'nun en iyi çözümü ile değiştir
                    if len(self.pop) > 0:
                        self.pop[0].solution = self.best_initial_solution.copy()
                        self.pop[0].target = self.problem.get_target(self.pop[0].solution)

            ALO_model = CustomALO(
                best_solution=best_position,
                epoch=self.ALO_epochs,
                pop_size=self.pop_size
            )
            ALO_result = ALO_model.solve(problem_dict)

            # En iyi sonucu güncelle
            if ALO_result.target.fitness < best_fitness:
                best_position = ALO_result.solution
                best_fitness = ALO_result.target.fitness
                print(f"ALO İyileştirme Sağladı - Yeni En İyi Fitness: {best_fitness:.6f}")
            else:
                print(f"ARO Sonucu Daha İyi Kaldı - En İyi Fitness: {best_fitness:.6f}")

        # Sonucu mealpy formatında döndür
        class HybridResult:
            def __init__(self, solution, fitness):
                self.solution = solution
                self.target = type('Target', (), {'fitness': fitness})()

        return HybridResult(best_position, best_fitness)

class MultiLevelThresholding:
    def __init__(self, image, num_thresholds=3):
        """
        Medical image multi-level thresholding using Hybrid ARO-ALO

        Parameters:
        image: numpy array - Input medical image
        num_thresholds: int - Number of threshold levels
        """
        self.image = image.astype(np.float64)
        self.num_thresholds = num_thresholds
        self.height, self.width = image.shape
        self.histogram = np.histogram(image, bins=256, range=(0, 255))[0]

    def otsu_fitness(self, thresholds):
        """
        Otsu's method fitness function for multi-level thresholding
        """
        thresholds = np.sort(thresholds)
        thresholds = np.clip(thresholds, 0, 255)

        # Add boundary thresholds
        full_thresholds = [0] + list(thresholds) + [255]

        total_pixels = self.height * self.width
        total_mean = np.sum(np.arange(256) * self.histogram) / total_pixels

        between_class_variance = 0

        for i in range(len(full_thresholds) - 1):
            start_idx = int(full_thresholds[i])
            end_idx = int(full_thresholds[i + 1])

            # Class probability
            w_i = np.sum(self.histogram[start_idx:end_idx]) / total_pixels

            if w_i > 0:
                # Class mean
                class_sum = np.sum(np.arange(start_idx, end_idx) * self.histogram[start_idx:end_idx])
                mu_i = class_sum / (w_i * total_pixels) if (w_i * total_pixels) > 0 else 0

                # Between-class variance contribution
                between_class_variance += w_i * (mu_i - total_mean) ** 2

        return -between_class_variance  # Negative because we want to maximize

    def apply_thresholding(self, thresholds):
        """
        Apply multi-level thresholding to the image
        """
        thresholds = np.sort(thresholds)
        segmented = np.zeros_like(self.image)

        # Apply thresholds
        segmented[self.image <= thresholds[0]] = 0
        for i in range(len(thresholds) - 1):
            mask = (self.image > thresholds[i]) & (self.image <= thresholds[i + 1])
            segmented[mask] = (i + 1) * (255 // (len(thresholds) + 1))
        segmented[self.image > thresholds[-1]] = 255

        return segmented.astype(np.uint8)

    def optimize_thresholds(self, pop_size=25, epochs=4, switch_ratio=0.6):
        """
        Optimize thresholds using Hybrid ARO-ALO

        Parameters:
        pop_size: int - Population size
        epochs: int - Total number of epochs
        switch_ratio: float - Ratio for ARO phase (0-1)
        """
        # Define bounds using FloatVar for mealpy v3.0+
        bounds = [FloatVar(lb=1.0, ub=255.0, name=f"threshold_{i}")
                  for i in range(self.num_thresholds)]

        # Define problem dictionary for mealpy
        problem_dict = {
            "bounds": bounds,
            "minmax": "min",  # We're minimizing (fitness function returns negative value)
            "obj_func": self.otsu_fitness
        }

        # Initialize Hybrid ARO-ALO
        hybrid_model = HybridARO_ALO(
            epoch=epochs,
            pop_size=pop_size,
            switch_ratio=switch_ratio
        )

        # Solve optimization problem
        g_best = hybrid_model.solve(problem_dict)

        return np.sort(g_best.solution), -g_best.target.fitness

def calculate_psnr(original, segmented):
    """Calculate Peak Signal-to-Noise Ratio"""
    mse = np.mean((original - segmented) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

def calculate_gradient_magnitude(image):
    gx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    gm = np.sqrt(gx ** 2 + gy ** 2)
    return gm

def calculate_phase_congruency_approx(image):
    image_blur = cv2.GaussianBlur(image, (5, 5), 1.0)
    gx = cv2.Sobel(image_blur, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(image_blur, cv2.CV_64F, 0, 1, ksize=3)
    gm = np.sqrt(gx ** 2 + gy ** 2)
    pc = gm / (gm + 10)  # epsilon to avoid division by zero
    return pc

def calculate_fsim(original, test):
    original = original.astype(np.float64)
    test = test.astype(np.float64)

    if original.max() > 1.0:
        original /= 255.0
        test /= 255.0

    # 1. Phase Congruency
    PC1 = calculate_phase_congruency_approx(original)
    PC2 = calculate_phase_congruency_approx(test)

    # 2. Gradient Magnitude
    GM1 = calculate_gradient_magnitude(original)
    GM2 = calculate_gradient_magnitude(test)

    # 3. Similarity Maps
    T1 = 0.01
    T2 = 0.02

    SPC = (2 * PC1 * PC2 + T1) / (PC1 ** 2 + PC2 ** 2 + T1)
    SGM = (2 * GM1 * GM2 + T2) / (GM1 ** 2 + GM2 ** 2 + T2)

    # 4. Combined Similarity
    PCm = np.maximum(PC1, PC2)
    FSIM_map = SPC * SGM
    numerator = np.sum(FSIM_map * PCm)
    denominator = np.sum(PCm)

    fsim_value = numerator / (denominator + 1e-10)
    return fsim_value


def calculate_ssim(original, segmented):
    """Calculate Structural Similarity Index"""
    return metrics.structural_similarity(original, segmented, data_range=255)

def friedman_ranking_test(data_matrix):
    """
    Perform Friedman ranking test
    data_matrix: rows are observations, columns are treatments/methods
    """
    n_observations, n_treatments = data_matrix.shape

    # Rank each row
    ranks = np.zeros_like(data_matrix)
    for i in range(n_observations):
        ranks[i] = rankdata(-data_matrix[i])  # Negative for descending order (higher is better)

    # Calculate rank sums
    rank_sums = np.sum(ranks, axis=0)

    # Friedman test statistic
    chi2_stat = (12 / (n_observations * n_treatments * (n_treatments + 1))) * \
                np.sum(rank_sums**2) - 3 * n_observations * (n_treatments + 1)

    # Degrees of freedom
    df = n_treatments - 1

    # P-value
    p_value = 1 - chi2.cdf(chi2_stat, df)

    return chi2_stat, p_value, rank_sums

def load_medical_images(healthy_folder,
                        max_images_per_class=6):
    """
    Load medical images from healthy and diseased folders
    """
    images = []
    labels = []
    filenames = []

    # Supported image extensions
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']

    # Load healthy images
    print(f"Loading healthy images from: {healthy_folder}")
    healthy_count = 0
    if os.path.exists(healthy_folder):
        for ext in extensions:
            for img_path in glob.glob(os.path.join(healthy_folder, ext)):
                if healthy_count >= max_images_per_class:
                    break
                try:
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        # Resize image to standard size for consistency
                        img = cv2.resize(img, (256,256))
                        images.append(img)
                        labels.append('Healthy')
                        filenames.append(os.path.basename(img_path))
                        healthy_count += 1
                        print(f"  Loaded: {os.path.basename(img_path)}")
                except Exception as e:
                    print(f"  Error loading {img_path}: {e}")
            if healthy_count >= max_images_per_class:
                break
    else:
        print(f"  Warning: Healthy folder not found: {healthy_folder}")


    if len(images) == 0:
        print("No images found! Please check your folder paths.")
        return None, None, None

    return images, labels, filenames

def main_analysis(healthy_folder="./healthy",
                  max_images_per_class=6):
    """
    Main analysis function with Hybrid ARO-ALO optimization
    """
    print("Medical Image Multi-level Thresholding Analysis with Hybrid ARO-ALO")
    print("=" * 70)

    # Load images from folders
    images, labels, filenames = load_medical_images(
        healthy_folder,
        max_images_per_class
    )

    if images is None:
        print("No images loaded. Exiting analysis.")
        return None, None, None

    # Different threshold levels to test
    threshold_levels = [2, 3, 4, 5, 6, 7,8,9,10,11,12]

    # Hybrid algorithm parameters
    hybrid_params = {
        'pop_size': 30,
        'epochs': 100,
        'switch_ratio': 0.2  # %60 ARO, %40 ALO
    }

    # Results storage
    results = {
        'Image': [],
        'Label': [],
        'Filename': [],
        'Thresholds': [],
        'PSNR': [],
        'SSIM': [],
        'FSIM': [],
        'Optimal_Thresholds': [],
        'Fitness_Value': [],
        'Algorithm': []
    }

    # Calculate number of subplot rows and columns
    n_images = len(images)
    n_cols = min(2, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols

    plt.figure(figsize=(n_cols * 6, n_rows * 3))

    for img_idx, (image, label, filename) in enumerate(zip(images, labels, filenames)):
        print(f"\nProcessing {filename} ({label})...")

        for thresh_level in threshold_levels:
            print(f"  Threshold levels: {thresh_level}")

            # Initialize thresholding with Hybrid ARO-ALO
            mt = MultiLevelThresholding(image, num_thresholds=thresh_level)

            # Optimize thresholds using Hybrid ARO-ALO
            optimal_thresholds, fitness_value = mt.optimize_thresholds(
                pop_size=hybrid_params['pop_size'],
                epochs=hybrid_params['epochs'],
                switch_ratio=hybrid_params['switch_ratio']
            )

            # Apply thresholding
            segmented = mt.apply_thresholding(optimal_thresholds)

            # Calculate metrics
            psnr = calculate_psnr(image, segmented)
            ssim = calculate_ssim(image, segmented)
            fsim = calculate_fsim(image, segmented)

            # Store results
            results['Image'].append(f"{label}_{img_idx+1}")
            results['Label'].append(label)
            results['Filename'].append(filename)
            results['Thresholds'].append(thresh_level)
            results['PSNR'].append(psnr)
            results['SSIM'].append(ssim)
            results['FSIM'].append(fsim)
            results['Optimal_Thresholds'].append(optimal_thresholds)
            results['Fitness_Value'].append(fitness_value)
            results['Algorithm'].append('Hybrid_ARO-ALO')

            print(f"    PSNR: {psnr:.2f}, SSIM: {ssim:.4f}, FSIM: {fsim:.4f}")
            print(f"    Optimal thresholds: {[f'{t:.1f}' for t in optimal_thresholds]}")

        # Visualize results for 3-level thresholding
        # Tek sefer optimizasyon - sonuçları sakla
        if img_idx == 0:  # İlk defa çalıştırılıyorsa storage listesi oluştur
            optimization_results = []

        # Visualize results for 3-level thresholding
        mt_viz = MultiLevelThresholding(image, num_thresholds=12)
        opt_thresh, _ = mt_viz.optimize_thresholds(
            pop_size=30,
            epochs=100,
            switch_ratio=0.8
        )
        segmented_viz = mt_viz.apply_thresholding(opt_thresh)

        # Sonuçları sakla
        optimization_results.append({
            'image': image,
            'segmented': segmented_viz,
            'thresholds': opt_thresh,
            'filename': filename,
            'label': label
        })

        # Plot original and segmented
        plt.subplot(n_rows, n_cols * 2, img_idx * 2 + 1)
        plt.imshow(image, cmap='plasma')
        plt.title('Original')
        plt.axis('off')

        plt.subplot(n_rows, n_cols * 2, img_idx * 2 + 2)
        im = plt.imshow(segmented_viz, cmap='plasma')
        plt.title('Hybrid ARO-ALO')
        plt.axis('off')
        # Scale bar ekleme
        plt.colorbar(im, ax=plt.gca(), fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


    # Calculate number of subplot rows and columns
    n_images = len(images)
    n_cols = min(1, n_images)
    n_rows = (n_images + n_cols - 1) // n_cols

    # Ayrı histogram analizi - saklanan sonuçları kullan
    print("\n" + "="*60)
    print("HISTOGRAM ANALYSIS")
    print("="*60)

    plt.figure(figsize=(n_cols * 6, n_rows * 3))
    image_number_count=1
    for idx, result in enumerate(optimization_results):
        image = result['image']
        segmented_viz = result['segmented']
        opt_thresh = result['thresholds']
        filename = result['filename']
        label = result['label']


        # Original 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 1)

        # Gradyan hesaplama (X ve Y yönünde)
        grad_x = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(image.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
        gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)

        # 2D histogram: Pixel değeri vs Gradyan büyüklüğü
        h, xedges, yedges = np.histogram2d(
            image.flatten(),
            gradient_magnitude.flatten(),
            bins=[64, 64],
            range=[[0, 255], [0, gradient_magnitude.max()]]
        )

        # 2D histogram görselleştirme
        extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
        im1 = plt.imshow(h.T, extent=extent, origin='lower', cmap='Blues', aspect='auto')
        plt.colorbar(im1, fraction=0.046, pad=0.04)
        plt.xlabel('Pixel Value')
        plt.ylabel('Gradient Magnitude')
        plt.title(f'Image {image_number_count}-2D Histogram\n) - Pixel vs Gradient')

        # Threshold çizgilerini ekleme
        for i, thresh in enumerate(opt_thresh):
            plt.axvline(x=thresh, color='red', linestyle='--', alpha=0.8, linewidth=2)

        # Segmented 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 2)

        # Segmented image için gradyan hesaplama
        grad_x_seg = cv2.Sobel(segmented_viz.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        grad_y_seg = cv2.Sobel(segmented_viz.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
        gradient_magnitude_seg = np.sqrt(grad_x_seg**2 + grad_y_seg**2)

        # 2D histogram: Pixel değeri vs Gradyan büyüklüğü (segmented)
        h_seg, xedges_seg, yedges_seg = np.histogram2d(
            segmented_viz.flatten(),
            gradient_magnitude_seg.flatten(),
            bins=[64, 64],
            range=[[0, 255], [0, gradient_magnitude_seg.max() if gradient_magnitude_seg.max() > 0 else 1]]
        )

        # 2D histogram görselleştirme (segmented)
        extent_seg = [xedges_seg[0], xedges_seg[-1], yedges_seg[0], yedges_seg[-1]]
        im2 = plt.imshow(h_seg.T, extent=extent_seg, origin='lower', cmap='Reds', aspect='auto')
        plt.colorbar(im2, fraction=0.046, pad=0.04)
        plt.xlabel('Pixel Value')
        plt.ylabel('Gradient Magnitude')
        plt.title(f'Segmented 2D Histogram\n-Hybrid ARO-ALO')
        image_number_count+=1

        # Threshold çizgilerini ekleme
        for i, thresh in enumerate(opt_thresh):
            plt.axvline(x=thresh, color='green', linestyle='--', alpha=0.8, linewidth=2,
                      label=f'T{i+1}: {thresh:.1f}' if idx == 0 else "")

        if idx == 0 and len(opt_thresh) > 0:
            plt.legend(fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()

    # Ek olarak: Pixel yoğunluk korelasyon analizi
    print("\n" + "="*60)
    print("PIXEL INTENSITY CORRELATION ANALYSIS")
    print("="*60)

    plt.figure(figsize=(n_cols * 12, n_rows * 6))

    image_number_count=1
    for idx, result in enumerate(optimization_results):
        image = result['image']
        segmented_viz = result['segmented']
        filename = result['filename']
        label = result['label']


        # Original vs Segmented pixel correlation
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 1)

        # Scatter plot: Original vs Segmented pixel values
        sample_indices = np.random.choice(image.size, min(5000, image.size), replace=False)
        orig_sample = image.flatten()[sample_indices]
        seg_sample = segmented_viz.flatten()[sample_indices]

        plt.hexbin(orig_sample, seg_sample, gridsize=50, cmap='Blues', alpha=0.7)
        plt.colorbar(fraction=0.046, pad=0.04)
        plt.xlabel('Original Pixel Value',fontdict={'size': 16})
        plt.ylabel('Segmented Pixel Value',fontdict={'size': 16})
        plt.title(f'Test Image {image_number_count} - Pixel Correlation\n - Original vs Segmented', fontdict={'size': 16})
        plt.plot([0, 255], [0, 255], 'r--', alpha=0.5, label='Perfect Correlation')
        plt.legend(fontsize="16")

        # Komşuluk analizi 2D histogram
        plt.subplot(n_rows, n_cols * 2, idx * 2 + 2)

        # Komşu pixel ilişkileri (horizontal)
        left_pixels = image[:, :-1].flatten()
        right_pixels = image[:, 1:].flatten()

        # 2D histogram: Sol pixel vs Sağ pixel
        h_neighbor, x_edges, y_edges = np.histogram2d(
            left_pixels, right_pixels,
            bins=[64, 64],
            range=[[0, 255], [0, 255]]
        )

        extent_neighbor = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
        im_neighbor = plt.imshow(h_neighbor.T, extent=extent_neighbor, origin='lower',
                              cmap='Greens', aspect='auto')
        plt.colorbar(im_neighbor, fraction=0.046, pad=0.04)
        plt.xlabel('Left Neighbor Pixel Value',fontdict={'size': 16})
        plt.ylabel('Right Neighbor Pixel Value',fontdict={'size': 16})
        plt.title(f'Test Image {image_number_count} - Spatial Correlation\n - Neighbor Pixel Analysis', fontdict={'size': 16})
        image_number_count+=1

    plt.tight_layout()
    plt.show()


    from scipy.stats import pearsonr
    # Orijinal vs Segment edilmiş piksel korelasyonu
    corr_orig_seg, _ = pearsonr(orig_sample, seg_sample)
    print(f"Original vs Segmented Pixel Pearson Correlation: {corr_orig_seg:.4f}")
    # Komşu pikseller arasındaki korelasyon (yatay)
    corr_neighbor, _ = pearsonr(left_pixels, right_pixels)
    print(f"Neighbor Pixel Pearson Correlation: {corr_neighbor:.4f}")


    from scipy.stats import spearmanr
    spearman_corr_orig_seg, _ = spearmanr(orig_sample, seg_sample)
    spearman_corr_neighbor, _ = spearmanr(left_pixels, right_pixels)
    print(f"Original vs Segmented Pixel Spearman Correlation: {spearman_corr_orig_seg:.4f}")
    print(f"Neighbor Pixel Spearman Correlation: {spearman_corr_neighbor:.4f}")

    from sklearn.metrics import mutual_info_score
    def mutual_info(x, y, bins=64):
        c_xy = np.histogram2d(x, y, bins)[0]
        mi = mutual_info_score(None, None, contingency=c_xy)
        return mi
    mi_orig_seg = mutual_info(orig_sample, seg_sample)
    mi_neighbor = mutual_info(left_pixels, right_pixels)
    print(f"Original vs Segmented Pixel Mutual Information: {mi_orig_seg:.4f}")
    print(f"Neighbor Pixel Mutual Information: {mi_neighbor:.4f}")

    # Create results DataFrame
    df = pd.DataFrame(results)
    print("\n" + "="*100)
    print("DETAILED RESULTS - HYBRID ARO-ALO OPTIMIZATION")
    print("="*100)
    print(df.to_string(index=False))

    # Statistical Analysis
    print("\n" + "="*100)
    print("HYBRID ARO-ALO PERFORMANCE ANALYSIS")
    print("="*100)

    # Performance comparison between healthy and diseased images
    healthy_df = df[df['Label'] == 'Healthy']
    #diseased_df = df[df['Label'] == 'Diseased']

    if len(healthy_df) > 0: #and len(diseased_df) > 0:
        print("\n--- HEALTHY vs DISEASED COMPARISON (Hybrid ARO-ALO) ---")

        healthy_psnr_avg = healthy_df.groupby('Thresholds')['PSNR'].mean()
        #diseased_psnr_avg = diseased_df.groupby('Thresholds')['PSNR'].mean()
        healthy_ssim_avg = healthy_df.groupby('Thresholds')['SSIM'].mean()
        #diseased_ssim_avg = diseased_df.groupby('Thresholds')['SSIM'].mean()
        healthy_fsim_avg = healthy_df.groupby('Thresholds')['FSIM'].mean()

        comparison_df = pd.DataFrame({
            'Thresholds': threshold_levels,
            'Healthy_PSNR': healthy_psnr_avg.values,
            #'Diseased_PSNR': diseased_psnr_avg.values,
            'Healthy_SSIM': healthy_ssim_avg.values,
            #'Diseased_SSIM': diseased_ssim_avg.values,
            'Healthy_FSIM': healthy_fsim_avg.values,
            #'PSNR_Improvement': healthy_psnr_avg.values - diseased_psnr_avg.values,
            #'SSIM_Improvement': healthy_ssim_avg.values - diseased_ssim_avg.values
        })

        print("\nHybrid ARO-ALO Performance Comparison:")
        print(comparison_df.round(4))

    # Friedman test analysis
    unique_images = df['Image'].unique()
    if len(unique_images) >= 3:

        psnr_matrix = np.zeros((len(unique_images), len(threshold_levels)))
        ssim_matrix = np.zeros((len(unique_images), len(threshold_levels)))
        fsim_matrix = np.zeros((len(unique_images), len(threshold_levels)))

        for i, img_name in enumerate(unique_images):
            img_data = df[df['Image'] == img_name]
            if len(img_data) == len(threshold_levels):
                psnr_matrix[i] = img_data['PSNR'].values
                ssim_matrix[i] = img_data['SSIM'].values
                fsim_matrix[i] = img_data['FSIM'].values

        # Friedman test for PSNR
        chi2_psnr, p_psnr, ranks_psnr = friedman_ranking_test(psnr_matrix)
        print(f"\nHybrid ARO-ALO PSNR Analysis:")
        print(f"Chi-square statistic: {chi2_psnr:.4f}")
        print(f"P-value: {p_psnr:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_psnr}")

        if p_psnr < 0.05:
            print("Significant difference found between threshold levels (PSNR)")
            best_psnr_level = threshold_levels[np.argmin(ranks_psnr)]
            print(f"Best performing threshold level (PSNR): {best_psnr_level}")
        else:
            print("No significant difference between threshold levels (PSNR)")

        # Friedman test for SSIM
        chi2_ssim, p_ssim, ranks_ssim = friedman_ranking_test(ssim_matrix)
        print(f"\nHybrid ARO-ALO SSIM Analysis:")
        print(f"Chi-square statistic: {chi2_ssim:.4f}")
        print(f"P-value: {p_ssim:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_ssim}")

        if p_ssim < 0.05:
            print("Significant difference found between threshold levels (SSIM)")
            best_ssim_level = threshold_levels[np.argmin(ranks_ssim)]
            print(f"Best performing threshold level (SSIM): {best_ssim_level}")
        else:
            print("No significant difference between threshold levels (SSIM)")


        # Friedman test for FSIM
        chi2_fsim, p_fsim, ranks_fsim = friedman_ranking_test(fsim_matrix)
        print(f"\nHybrid ARO-ALO FSIM Analysis:")
        print(f"Chi-square statistic: {chi2_fsim:.4f}")
        print(f"P-value: {p_fsim:.4f}")
        print(f"Rank sums for threshold levels {threshold_levels}: {ranks_fsim}")
        if p_fsim < 0.05:
            print("Significant difference found between threshold levels (FSIM)")
            best_fsim_level = threshold_levels[np.argmin(ranks_fsim)]
            print(f"Best performing threshold level (FSIM): {best_fsim_level}")
        else:
            print("No significant difference between threshold levels (FSIM)")

    else:
        print(f"Not enough images ({len(unique_images)}) for Friedman test.")
        chi2_psnr, p_psnr, ranks_psnr = None, None, None
        chi2_ssim, p_ssim, ranks_ssim = None, None, None
        chi2_fsim, p_fsim, ranks_fsim = None, None, None

    # Summary Statistics
    print("\n" + "="*100)
    print("HYBRID ARO-ALO SUMMARY STATISTICS")
    print("="*100)

    if len(df) > 0:
        summary_stats = df.groupby(['Label', 'Thresholds']).agg({
            'PSNR': ['mean', 'std', 'min', 'max'],
            'SSIM': ['mean', 'std', 'min', 'max'],
            'FSIM': ['mean', 'std', 'min', 'max'],
            'Fitness_Value': ['mean', 'std', 'min', 'max']
        }).round(4)

        print(summary_stats)

        # Plot enhanced comparison charts
        plt.figure(figsize=(20, 12))

        # PSNR comparison
        plt.subplot(3, 4, 1)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            psnr_means = label_data.groupby('Thresholds')['PSNR'].mean()
            psnr_stds = label_data.groupby('Thresholds')['PSNR'].std()
            plt.errorbar(threshold_levels, psnr_means, yerr=psnr_stds,
                        marker='o', label=f'{label} (Hybrid ARO-ALO)',
                        linewidth=2, capsize=5)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average PSNR')
        plt.title('Hybrid ARO-ALO: PSNR Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # SSIM comparison
        plt.subplot(3, 4, 2)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            ssim_means = label_data.groupby('Thresholds')['SSIM'].mean()
            ssim_stds = label_data.groupby('Thresholds')['SSIM'].std()
            plt.errorbar(threshold_levels, ssim_means, yerr=ssim_stds,
                        marker='s', label=f'{label} (Hybrid ARO-ALO)',
                        linewidth=2, capsize=5)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average SSIM')
        plt.title('Hybrid ARO-ALO: SSIM Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)



        # FSIM comparison
        plt.subplot(3, 4, 3)
        for label in df['Label'].unique():
          label_data = df[df['Label'] == label]
          fsim_means = label_data.groupby('Thresholds')['FSIM'].mean()
          fsim_stds = label_data.groupby('Thresholds')['FSIM'].std()
          plt.errorbar(threshold_levels, fsim_means, yerr=fsim_stds,
                       marker='^', label=f'{label} (Hybrid ARO-ALO)',
                       linewidth=2, capsize=5)
          plt.xlabel('Threshold Levels')
          plt.ylabel('Average FSIM')
          plt.title('Hybrid ARO-ALO: FSIM Performance')
          plt.legend()
          plt.grid(True, alpha=0.3)

        # Fitness value comparison
        plt.subplot(3, 4, 4)
        for label in df['Label'].unique():
            label_data = df[df['Label'] == label]
            fitness_means = label_data.groupby('Thresholds')['Fitness_Value'].mean()
            plt.plot(threshold_levels, fitness_means, marker='^',
                    label=f'{label} (Hybrid ARO-ALO)', linewidth=2)
        plt.xlabel('Threshold Levels')
        plt.ylabel('Average Fitness Value')
        plt.title('Hybrid ARO-ALO: Fitness Performance')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Box plots
        plt.subplot(3, 4, 5)
        df.boxplot(column='PSNR', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('PSNR Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 6)
        df.boxplot(column='SSIM', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('SSIM Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 7)
        df.boxplot(column='FSIM', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('FSIM Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.subplot(3, 4, 8)
        df.boxplot(column='Fitness_Value', by=['Label', 'Thresholds'], ax=plt.gca())
        plt.title('Fitness Distribution (Hybrid ARO-ALO)')
        plt.xticks(rotation=45)
        plt.suptitle('')

        plt.tight_layout()
        plt.show()

    return df, (chi2_psnr, p_psnr, ranks_psnr), (chi2_ssim, p_ssim, ranks_ssim), (chi2_fsim, p_fsim, ranks_fsim)

if __name__ == "__main__":
    # Dataset Path
    healthy_folder = ""


    print("Starting Hybrid ARO-ALO Medical Image Segmentation Analysis...")
    print("Algorithm Configuration:")
    print("- Phase 1: ARO (60% of epochs) - Exploration")
    print("- Phase 2: ALO (40% of epochs) - Exploitation")
    print("- Population Size: 30")
    print("- Total Epochs: 100-150")
    print("-" * 60)

    # Analizi çalıştır
    results_df, psnr_test, ssim_test, fsim_test = main_analysis(
        healthy_folder=healthy_folder,
        max_images_per_class=100
    )

    if results_df is not None:
        print("\nHybrid ARO-ALO Analysis completed successfully!")
        print("Results saved in 'results_df' DataFrame")

        # Sonuçları CSV dosyasına kaydet
        results_df.to_csv('hybrid_ARO_ALO_segmentation_results.csv', index=False)
        print("Results also saved to 'hybrid_ARO_ALO_segmentation_results.csv'")

        # Algoritma performans özeti
        print("\n" + "="*80)
        print("HYBRID ARO-ALO ALGORITHM PERFORMANCE SUMMARY")
        print("="*80)

        avg_psnr = results_df['PSNR'].mean()
        avg_ssim = results_df['SSIM'].mean()
        avg_fsim = results_df['FSIM'].mean()
        avg_fitness = results_df['Fitness_Value'].mean()

        print(f"Overall Average PSNR: {avg_psnr:.4f}")
        print(f"Overall Average SSIM: {avg_ssim:.4f}")
        print(f"Overall Average FSIM: {avg_fsim:.4f}")
        print(f"Overall Average Fitness: {avg_fitness:.6f}")

        # En iyi performans gösteren threshold seviyesi
        best_threshold_psnr = results_df.groupby('Thresholds')['PSNR'].mean().idxmax()
        best_threshold_ssim = results_df.groupby('Thresholds')['SSIM'].mean().idxmax()
        best_threshold_fsim = results_df.groupby('Thresholds')['FSIM'].mean().idxmax()

        print(f"Best Threshold Level for PSNR: {best_threshold_psnr}")
        print(f"Best Threshold Level for SSIM: {best_threshold_ssim}")
        print(f"Best Threshold Level for FSIM: {best_threshold_fsim}")

    else:
        print("Analysis failed - no images were loaded.")